# Pelatihan Deep Learning Multi-Task BiLSTM: Estimasi Tekanan Darah dari Sinyal ECG

**Notebook untuk Kaggle** — Self-contained, semua kode (preprocessing, model, loss, training) sudah terinline.

### Ringkasan Arsitektur & Metode
| Komponen | Detail |
|---|---|
| **Dataset** | Kachuee et al. (2016) — Cuff-Less Blood Pressure Dataset (MIMIC-II) |
| **Preprocessing** | IIR Notch 50 Hz → Butterworth BPF 0.5–40 Hz → Min-Max Normalization [0,1] |
| **Model** | 2-Layer Bidirectional LSTM (hidden=256, output=512) + 3 Task-Specific FC Heads |
| **Loss** | Adaptive Weight Learning Multi-Task MAE (Fan et al., 2021) |
| **Target** | SBP, DBP, MAP (mmHg) simultan |
| **Acuan Utama** | Fan et al. (2021, Sensors) — "An Adaptive Weight Learning-Based Multitask Deep Network" |

### Petunjuk di Kaggle
1. Buat notebook baru di Kaggle
2. **Add Data** → cari `mkachuee/BloodPressureDataset` → tambahkan
3. **Settings** → Accelerator → pilih **GPU T4 x2** atau **P100**
4. **Run All** — notebook ini akan memproses seluruh dataset lalu melatih model

---
## 1. Setup & Instalasi

In [ ]:
import os
import glob
import time
import json
import gc
import numpy as np
import scipy.io
import matplotlib.pyplot as plt
from scipy.signal import iirnotch, butter, filtfilt, find_peaks
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

# ── Deteksi akselerator ──────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f'✅ GPU terdeteksi: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
    print('✅ Apple Silicon MPS terdeteksi')
else:
    DEVICE = torch.device('cpu')
    print('⚠️ Tidak ada GPU — menggunakan CPU (akan sangat lambat!)')

print(f'PyTorch: {torch.__version__} | Device: {DEVICE}')

---
## 2. Hyperparameter & Konfigurasi

Semua parameter sesuai paper acuan **Fan et al. (2021, Sensors)**.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#                        HYPERPARAMETER PELATIHAN
# ══════════════════════════════════════════════════════════════════════════════

# -- Preprocessing --
FS = 125.0              # Sampling rate dataset Kachuee (Hz)
EPOCH_SEC = 10.0        # Durasi 1 epoch/window (detik)
SEQ_LEN = int(EPOCH_SEC * FS)  # = 1250 sampel per epoch

NOTCH_FREQ = 50.0       # Frekuensi notch PLN (Hz)
NOTCH_Q = 30.0          # Quality factor notch filter
BP_LOWCUT = 0.5         # Bandpass low cutoff (Hz)
BP_HIGHCUT = 40.0       # Bandpass high cutoff (Hz)
BP_ORDER = 2            # Orde Butterworth (efektif orde 4 via filtfilt)

# -- Model BiLSTM (Tabel 1 Fan et al., 2021) --
INPUT_SIZE = 1          # 1 channel ECG
HIDDEN_SIZE = 256       # BiLSTM hidden units (output = 2*256 = 512)
NUM_LAYERS = 2          # Jumlah layer BiLSTM bertumpuk
DROPOUT = 0.3           # Dropout antar layer LSTM
FC_DROPOUT = 0.2        # Dropout di FC heads

# -- Training (Fan et al., 2021) --
BATCH_SIZE = 128
NUM_EPOCHS = 30
LEARNING_RATE = 8e-4    # lr = 0.0008
WEIGHT_DECAY = 1e-5     # L2 regularization
GRAD_CLIP = 5.0         # Gradient clipping max norm
SCHEDULER_FACTOR = 0.9  # ReduceLROnPlateau factor
SCHEDULER_PATIENCE = 4  # ReduceLROnPlateau patience

# -- Data Split --
TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10
SEED = 42

# -- Filter kualitas BP (rentang fisiologis valid) --
SBP_RANGE = (50.0, 220.0)   # mmHg
DBP_RANGE = (30.0, 140.0)   # mmHg

print('Hyperparameter siap.')
print(f'  Seq Length : {SEQ_LEN} sampel ({EPOCH_SEC}s @ {FS} Hz)')
print(f'  Batch Size : {BATCH_SIZE}')
print(f'  Epochs     : {NUM_EPOCHS}')
print(f'  LR         : {LEARNING_RATE}')

---
## 3. Fungsi Preprocessing (Inline)

Mereplikasi fungsi dari `src/preprocessing/` agar notebook mandiri di Kaggle.

**Pipeline**: Raw ECG → Notch 50 Hz → Bandpass 0.5–40 Hz → Min-Max [0,1]

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#                    FUNGSI PREPROCESSING (SELF-CONTAINED)
# ══════════════════════════════════════════════════════════════════════════════

def iir_notch_filter(signal, fs=125.0, f0=50.0, q=30.0):
    """
    IIR Notch Filter — Eliminasi interferensi frekuensi listrik PLN 50 Hz.
    Metode: Zero-Phase Filtering (filtfilt) sesuai Alhamid (2026) & Hakim (2025).
    """
    if signal.size == 0:
        return signal
    b, a = iirnotch(w0=f0, Q=q, fs=fs)
    return filtfilt(b, a, signal, axis=0)


def butterworth_bandpass_filter(signal, fs=125.0, lowcut=0.5, highcut=40.0, order=2):
    """
    Butterworth Bandpass Filter 0.5–40 Hz.
    Zero-Phase Filtering (filtfilt) → orde efektif = 2 * order = 4.
    """
    if signal.size == 0:
        return signal
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    if high >= 1.0:
        high = 0.99
    b, a = butter(N=order, Wn=[low, high], btype='bandpass')
    return filtfilt(b, a, signal, axis=0)


def minmax_scale(signal, eps=1e-8):
    """
    Min-Max Normalization [0, 1] sesuai Fan et al. (2021) & Li et al. (2020).
    Formula: x_norm = (signal - x_min) / (x_max - x_min + eps)
    """
    if signal.size == 0:
        return signal
    x_min = np.min(signal)
    x_max = np.max(signal)
    range_val = x_max - x_min
    if range_val < eps:
        range_val = 1.0
    return (signal - x_min) / range_val


def preprocess_ecg(signal, fs=125.0):
    """Pipeline preprocessing lengkap: Notch → Bandpass → MinMax."""
    signal = iir_notch_filter(signal, fs=fs, f0=NOTCH_FREQ, q=NOTCH_Q)
    signal = butterworth_bandpass_filter(signal, fs=fs, lowcut=BP_LOWCUT, highcut=BP_HIGHCUT, order=BP_ORDER)
    signal = minmax_scale(signal)
    return signal


def compute_bp_per_beat(abp_signal, fs=125.0):
    """
    Menghitung SBP, DBP, MAP dari sinyal ABP menggunakan deteksi puncak per beat.
    Metode beat-by-beat sesuai standar klinis (bukan global max/min).
    """
    min_peak_distance = int(0.3 * fs)  # ~37 sampel (HR ≤ 200 bpm)
    min_peak_height = np.percentile(abp_signal, 50)

    systolic_peaks, _ = find_peaks(abp_signal, distance=min_peak_distance, height=min_peak_height)
    diastolic_troughs, _ = find_peaks(-abp_signal, distance=min_peak_distance)

    sbp = float(np.mean(abp_signal[systolic_peaks])) if len(systolic_peaks) > 0 else float(np.max(abp_signal))
    dbp = float(np.mean(abp_signal[diastolic_troughs])) if len(diastolic_troughs) > 0 else float(np.min(abp_signal))
    map_val = float(np.mean(abp_signal))

    is_valid = (
        len(systolic_peaks) > 0
        and SBP_RANGE[0] <= sbp <= SBP_RANGE[1]
        and DBP_RANGE[0] <= dbp <= DBP_RANGE[1]
    )
    return sbp, dbp, map_val, is_valid

print('✅ Fungsi preprocessing siap.')

---
## 4. Pemuatan & Preprocessing Seluruh Dataset Kachuee

Memuat semua file `part_*.mat`, melakukan epoching 10 detik, preprocessing, dan ekstraksi ground truth BP.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#                     PEMUATAN & PREPROCESSING DATASET
# ══════════════════════════════════════════════════════════════════════════════

# Path dataset di Kaggle (sesuaikan jika berbeda)
KAGGLE_DATA_DIR = '/kaggle/input/bloodpressuredataset'

# Fallback: coba path alternatif jika path utama tidak ditemukan
if not os.path.exists(KAGGLE_DATA_DIR):
    alt_paths = [
        '/kaggle/input/BloodPressureDataset',
        '/kaggle/input/bloodpressuredataset/BloodPressureDataset',
    ]
    for p in alt_paths:
        if os.path.exists(p):
            KAGGLE_DATA_DIR = p
            break
    else:
        # Jika dijalankan di lokal, gunakan kagglehub
        try:
            import kagglehub
            KAGGLE_DATA_DIR = kagglehub.dataset_download('mkachuee/BloodPressureDataset')
        except ImportError:
            raise FileNotFoundError(
                'Dataset tidak ditemukan. Di Kaggle, pastikan sudah Add Data "mkachuee/BloodPressureDataset".'
            )

# Cari semua file .mat
mat_files = sorted(glob.glob(os.path.join(KAGGLE_DATA_DIR, '**', '*.mat'), recursive=True))
print(f'Direktori dataset: {KAGGLE_DATA_DIR}')
print(f'Jumlah file .mat ditemukan: {len(mat_files)}')
for f in mat_files:
    size_mb = os.path.getsize(f) / 1024**2
    print(f'  • {os.path.basename(f)} ({size_mb:.1f} MB)')

In [ ]:
def load_mat_file(file_path):
    """Membaca file .mat (MATLAB v5 atau HDF5 v7.3)."""
    try:
        mat = scipy.io.loadmat(file_path)
        keys = [k for k in mat.keys() if not k.startswith('__')]
        return mat[keys[0]]
    except Exception:
        import h5py
        with h5py.File(file_path, 'r') as f:
            return np.array(f[list(f.keys())[0]])


def extract_record_matrix(cell_data, index):
    """Ekstrak matriks sinyal [3, N_samples] dari cell array."""
    if isinstance(cell_data, np.ndarray) and cell_data.dtype == object:
        raw = cell_data[0, index]
    else:
        raw = cell_data[index]
    matrix = np.array(raw, dtype=np.float64)
    if matrix.ndim == 2 and matrix.shape[0] > matrix.shape[1] and matrix.shape[1] == 3:
        matrix = matrix.T
    return matrix


def get_num_records(cell_data):
    """Hitung jumlah rekaman dalam cell array."""
    if isinstance(cell_data, np.ndarray) and cell_data.dtype == object:
        return cell_data.shape[1]
    return cell_data.shape[0]


print('✅ Fungsi pemuat dataset siap.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#    EPOCHING + PREPROCESSING + EKSTRAKSI GROUND TRUTH BP (SELURUH DATASET)
# ══════════════════════════════════════════════════════════════════════════════

all_X = []  # ECG epochs setelah preprocessing [N, 1250]
all_y = []  # Ground truth BP [N, 3] = [SBP, DBP, MAP]

total_records = 0
total_epochs_valid = 0
total_epochs_skipped = 0

for mat_idx, mat_path in enumerate(mat_files):
    part_name = os.path.basename(mat_path)
    print(f'\n{"="*70}')
    print(f'[{mat_idx+1}/{len(mat_files)}] Memproses: {part_name}')
    print(f'{"="*70}')

    cell_data = load_mat_file(mat_path)
    n_records = get_num_records(cell_data)
    total_records += n_records
    print(f'  Jumlah rekaman: {n_records:,}')

    part_valid = 0
    part_skipped = 0

    for rec_idx in tqdm(range(n_records), desc=f'  {part_name}', leave=True):
        try:
            matrix = extract_record_matrix(cell_data, rec_idx)
        except Exception:
            part_skipped += 1
            continue

        # Validasi dimensi minimal: [3, N_samples] dengan N >= SEQ_LEN
        if matrix.ndim != 2 or matrix.shape[0] < 3 or matrix.shape[1] < SEQ_LEN:
            part_skipped += 1
            continue

        ecg_full = matrix[2, :]   # ECG Lead II
        abp_full = matrix[1, :]   # ABP invasif
        n_samples_total = ecg_full.shape[0]

        # Epoching non-overlapping 10 detik
        n_epochs = n_samples_total // SEQ_LEN

        for ep in range(n_epochs):
            start = ep * SEQ_LEN
            end = start + SEQ_LEN

            ecg_epoch = ecg_full[start:end]
            abp_epoch = abp_full[start:end]

            # Cek validitas sinyal (bukan flat line atau NaN)
            if np.isnan(ecg_epoch).any() or np.isnan(abp_epoch).any():
                part_skipped += 1
                continue
            if np.std(ecg_epoch) < 1e-6 or np.std(abp_epoch) < 1e-6:
                part_skipped += 1
                continue

            # Hitung ground truth BP per epoch (beat-by-beat)
            sbp, dbp, map_val, is_valid = compute_bp_per_beat(abp_epoch, fs=FS)

            if not is_valid:
                part_skipped += 1
                continue

            # Preprocessing ECG
            ecg_clean = preprocess_ecg(ecg_epoch, fs=FS)

            all_X.append(ecg_clean)
            all_y.append([sbp, dbp, map_val])
            part_valid += 1

    total_epochs_valid += part_valid
    total_epochs_skipped += part_skipped
    print(f'  ✅ {part_name}: {part_valid:,} epoch valid | {part_skipped:,} epoch dilewati')

    # Bebaskan memori setiap part selesai
    del cell_data
    gc.collect()

# Konversi ke numpy array
X_all = np.array(all_X, dtype=np.float32)  # [N, 1250]
y_all = np.array(all_y, dtype=np.float32)  # [N, 3]

print(f'\n{"="*70}')
print(f'RINGKASAN DATASET SETELAH PREPROCESSING')
print(f'{"="*70}')
print(f'  Total rekaman sumber    : {total_records:,}')
print(f'  Total epoch valid       : {total_epochs_valid:,}')
print(f'  Total epoch dilewati    : {total_epochs_skipped:,}')
print(f'  Shape X (ECG)           : {X_all.shape}')
print(f'  Shape y (SBP,DBP,MAP)   : {y_all.shape}')
print(f'  Memori X                : {X_all.nbytes / 1024**2:.1f} MB')
print(f'\n  Statistik Ground Truth BP:')
for i, name in enumerate(['SBP', 'DBP', 'MAP']):
    vals = y_all[:, i]
    print(f'    {name}: mean={vals.mean():.1f} | std={vals.std():.1f} | '
          f'min={vals.min():.1f} | max={vals.max():.1f} mmHg')

---
## 5. Visualisasi Distribusi Ground Truth BP

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
bp_names = ['SBP (Sistolik)', 'DBP (Diastolik)', 'MAP (Mean Arterial)']
colors = ['#e74c3c', '#3498db', '#2ecc71']

for i, (name, color) in enumerate(zip(bp_names, colors)):
    vals = y_all[:, i]
    axes[i].hist(vals, bins=50, density=True, color=color, alpha=0.7, edgecolor='white')
    axes[i].axvline(vals.mean(), color='black', linestyle='--', linewidth=1.5,
                    label=f'Mean: {vals.mean():.1f} mmHg')
    axes[i].set_title(f'Distribusi {name}', fontsize=13, fontweight='bold')
    axes[i].set_xlabel('Tekanan Darah (mmHg)')
    axes[i].set_ylabel('Densitas')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.suptitle(f'Distribusi Ground Truth Tekanan Darah — {len(y_all):,} Epoch Total',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 6. Definisi Model (Self-Contained)

### Arsitektur Multi-Task BiLSTM (Fan et al., 2021)

```
ECG [batch, 1250, 1]
  │
  ▼
BiLSTM Layer 1 (hidden=256, bidirectional → 512)
  │ dropout=0.3
  ▼
BiLSTM Layer 2 (hidden=256, bidirectional → 512)
  │
  ▼
Terminal State Extraction → [batch, 512]
  │
  ├─→ SBP Head: FC(512→512)→ReLU→Drop→FC(512→256)→ReLU→Drop→FC(256→1)
  ├─→ DBP Head: FC(512→512)→ReLU→Drop→FC(512→256)→ReLU→Drop→FC(256→1)
  └─→ MAP Head: FC(512→512)→ReLU→Drop→FC(512→256)→ReLU→Drop→FC(256→1)
```

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#                        DEFINISI MODEL (INLINE)
# ══════════════════════════════════════════════════════════════════════════════

class TaskSpecificHead(nn.Module):
    """
    Fully Connected Regression Head untuk masing-masing target BP.
    Fan et al. (2021) Section 2.3 & Table 1:
      Hidden 1: 512 → ReLU → Dropout
      Hidden 2: 256 → ReLU → Dropout
      Output  : 1 (Linear murni)
    """
    def __init__(self, in_features=512, hidden1=512, hidden2=256, dropout=0.2):
        super().__init__()
        layers = [nn.Linear(in_features, hidden1), nn.ReLU()]
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        layers.extend([nn.Linear(hidden1, hidden2), nn.ReLU()])
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        layers.append(nn.Linear(hidden2, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class ECGMultiTaskBiLSTM(nn.Module):
    """
    Model Multi-Task BiLSTM untuk estimasi simultan SBP, DBP, MAP dari sinyal ECG.
    Sesuai Tabel 1 & Section 2.3 Fan et al. (2021, Sensors).
    """
    def __init__(self, input_size=1, hidden_size=256, num_layers=2,
                 lstm_dropout=0.3, fc_dropout=0.2):
        super().__init__()
        self.hidden_size = hidden_size

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=lstm_dropout if num_layers > 1 else 0.0,
        )

        encoder_dim = hidden_size * 2  # 512
        self.sbp_head = TaskSpecificHead(encoder_dim, encoder_dim, encoder_dim // 2, fc_dropout)
        self.dbp_head = TaskSpecificHead(encoder_dim, encoder_dim, encoder_dim // 2, fc_dropout)
        self.map_head = TaskSpecificHead(encoder_dim, encoder_dim, encoder_dim // 2, fc_dropout)

    def extract_features(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(-1)  # [batch, seq, 1]
        lstm_out, _ = self.lstm(x)
        # Terminal state: forward akhir + backward awal (Fan et al., 2021)
        fwd_last = lstm_out[:, -1, :self.hidden_size]
        bwd_first = lstm_out[:, 0, self.hidden_size:]
        return torch.cat([fwd_last, bwd_first], dim=-1)  # [batch, 512]

    def forward(self, x):
        features = self.extract_features(x)
        return self.sbp_head(features), self.dbp_head(features), self.map_head(features)

    def predict_all(self, x):
        sbp, dbp, map_val = self.forward(x)
        return torch.cat([sbp, dbp, map_val], dim=-1)  # [batch, 3]


# Hitung total parameter
_test_model = ECGMultiTaskBiLSTM(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, DROPOUT, FC_DROPOUT)
total_params = sum(p.numel() for p in _test_model.parameters())
trainable_params = sum(p.numel() for p in _test_model.parameters() if p.requires_grad)
print(f'✅ Model ECGMultiTaskBiLSTM berhasil didefinisikan.')
print(f'   Total parameter     : {total_params:,}')
print(f'   Trainable parameter : {trainable_params:,}')
del _test_model

---
## 7. Adaptive Multi-Task Loss (Fan et al., 2021)

Formulasi (Section 2.3 & Algorithm 1):

$$L_{total} = \theta_s \cdot L_{SBP} + \theta_d \cdot L_{DBP} + \theta_m \cdot L_{MAP}$$

Bobot $\theta$ diperbarui secara adaptif setiap akhir epoch validasi berdasarkan tren loss (Eq. 8–10).

In [ ]:
class AdaptiveMultiTaskLoss(nn.Module):
    """
    Adaptive Weight Learning Multi-Task Loss sesuai Fan et al. (2021) Algorithm 1.
    """
    def __init__(self):
        super().__init__()
        # Inisialisasi bobot tugas: theta_i = 1/3
        self.register_buffer('weights', torch.tensor([1/3, 1/3, 1/3], dtype=torch.float32))
        self.prev_val_mean = None
        self.prev_val_std = None

    def forward(self, sbp_pred, dbp_pred, map_pred, targets):
        sbp_true = targets[:, 0:1]
        dbp_true = targets[:, 1:2]
        map_true = targets[:, 2:3]

        sbp_pred = sbp_pred.view_as(sbp_true)
        dbp_pred = dbp_pred.view_as(dbp_true)
        map_pred = map_pred.view_as(map_true)

        loss_sbp = F.l1_loss(sbp_pred, sbp_true)
        loss_dbp = F.l1_loss(dbp_pred, dbp_true)
        loss_map = F.l1_loss(map_pred, map_true)

        total = self.weights[0] * loss_sbp + self.weights[1] * loss_dbp + self.weights[2] * loss_map

        metrics = {
            'loss_total': total.item(),
            'mae_sbp': F.l1_loss(sbp_pred, sbp_true).item(),
            'mae_dbp': F.l1_loss(dbp_pred, dbp_true).item(),
            'mae_map': F.l1_loss(map_pred, map_true).item(),
        }
        return total, metrics

    def update_weights(self, val_sbp_losses, val_dbp_losses, val_map_losses):
        """Pembaruan bobot adaptif Eq. 8-10 Fan et al. (2021)."""
        eps = 1e-8
        cur_mean = np.array([np.mean(val_sbp_losses), np.mean(val_dbp_losses), np.mean(val_map_losses)])
        cur_std = np.array([np.std(val_sbp_losses), np.std(val_dbp_losses), np.std(val_map_losses)])

        if self.prev_val_mean is not None:
            t_mean = np.abs(cur_mean - self.prev_val_mean) / (cur_mean + eps)
            t_std = np.abs(cur_std - self.prev_val_std) / (cur_std + eps)
            raw = t_mean * t_std * (cur_mean + cur_std)
            s = np.sum(raw)
            if s > eps:
                norm_w = raw / s
            else:
                norm_w = np.array([1/3, 1/3, 1/3])
            self.weights.copy_(torch.tensor(norm_w, dtype=torch.float32, device=self.weights.device))

        self.prev_val_mean = cur_mean
        self.prev_val_std = cur_std

print('✅ AdaptiveMultiTaskLoss siap.')

---
## 8. Dataset & DataLoader PyTorch

In [ ]:
class ECGBPDataset(Dataset):
    """Dataset PyTorch untuk pasangan ECG [1250] → BP [SBP, DBP, MAP]."""
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float().unsqueeze(-1)  # [N, 1250, 1]
        self.y = torch.from_numpy(y).float()                 # [N, 3]

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# Buat dataset
dataset = ECGBPDataset(X_all, y_all)

# Split: 80% Train / 10% Val / 10% Test
total_len = len(dataset)
train_len = int(TRAIN_RATIO * total_len)
val_len = int(VAL_RATIO * total_len)
test_len = total_len - train_len - val_len

train_set, val_set, test_set = random_split(
    dataset, [train_len, val_len, test_len],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          drop_last=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=True)

print(f'\n📊 Pembagian Dataset (Seed={SEED}):')
print(f'   Train : {len(train_set):>8,} ({TRAIN_RATIO*100:.0f}%)')
print(f'   Val   : {len(val_set):>8,} ({VAL_RATIO*100:.0f}%)')
print(f'   Test  : {len(test_set):>8,} ({TEST_RATIO*100:.0f}%)')
print(f'   Total : {total_len:>8,}')
print(f'   Batch : {BATCH_SIZE} | Train steps/epoch: {len(train_loader)}')

---
## 9. Inisialisasi Model, Optimizer, Scheduler

In [ ]:
# Model
model = ECGMultiTaskBiLSTM(
    input_size=INPUT_SIZE,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    lstm_dropout=DROPOUT,
    fc_dropout=FC_DROPOUT
).to(DEVICE)

# Loss
criterion = AdaptiveMultiTaskLoss().to(DEVICE)

# Optimizer (AdamW)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# LR Scheduler (Fan et al., 2021: factor=0.9, patience=4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=SCHEDULER_FACTOR, patience=SCHEDULER_PATIENCE
)

print(f'\n🏗️ Model di {DEVICE}')
print(f'   Optimizer : AdamW (lr={LEARNING_RATE}, wd={WEIGHT_DECAY})')
print(f'   Scheduler : ReduceLROnPlateau (factor={SCHEDULER_FACTOR}, patience={SCHEDULER_PATIENCE})')
print(f'   Grad Clip : max_norm={GRAD_CLIP}')

---
## 10. Training Loop

In [ ]:
def train_one_epoch(model, criterion, optimizer, loader, device):
    """Satu epoch training."""
    model.train()
    total_loss, mae_s, mae_d, mae_m, n = 0, 0, 0, 0, 0
    for x_batch, y_batch in loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        sbp, dbp, mapp = model(x_batch)
        loss, metrics = criterion(sbp, dbp, mapp, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()
        total_loss += metrics['loss_total']
        mae_s += metrics['mae_sbp']
        mae_d += metrics['mae_dbp']
        mae_m += metrics['mae_map']
        n += 1
    return {
        'loss': total_loss / max(1, n),
        'mae_sbp': mae_s / max(1, n),
        'mae_dbp': mae_d / max(1, n),
        'mae_map': mae_m / max(1, n),
    }


@torch.no_grad()
def evaluate(model, criterion, loader, device, update_weights=False):
    """Evaluasi pada validation / test set."""
    model.eval()
    total_loss, n = 0, 0
    sbp_losses, dbp_losses, map_losses = [], [], []
    for x_batch, y_batch in loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        sbp, dbp, mapp = model(x_batch)
        loss, metrics = criterion(sbp, dbp, mapp, y_batch)
        total_loss += metrics['loss_total']
        sbp_losses.append(metrics['mae_sbp'])
        dbp_losses.append(metrics['mae_dbp'])
        map_losses.append(metrics['mae_map'])
        n += 1
    if update_weights:
        criterion.update_weights(sbp_losses, dbp_losses, map_losses)
    return {
        'loss': total_loss / max(1, n),
        'mae_sbp': float(np.mean(sbp_losses)),
        'mae_dbp': float(np.mean(dbp_losses)),
        'mae_map': float(np.mean(map_losses)),
    }

print('✅ Fungsi training & evaluasi siap.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#                         PROSES PELATIHAN UTAMA
# ══════════════════════════════════════════════════════════════════════════════

CHECKPOINT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
best_model_path = os.path.join(CHECKPOINT_DIR, 'best_model.pt')

history = {
    'train_loss': [], 'val_loss': [],
    'val_mae_sbp': [], 'val_mae_dbp': [], 'val_mae_map': [],
    'lr': [], 'weights': []
}

best_val_mae = float('inf')

print('=' * 80)
print('  PELATIHAN DEEP LEARNING: MULTI-TASK BiLSTM (ECG → SBP/DBP/MAP)')
print('  Referensi Parameter: Fan et al. (2021, Sensors)')
print('=' * 80)
print(f'  Device         : {DEVICE}')
print(f'  Train samples  : {len(train_set):,}')
print(f'  Batch size     : {BATCH_SIZE}')
print(f'  Epochs         : {NUM_EPOCHS}')
print(f'  LR             : {LEARNING_RATE}')
print('=' * 80)

start_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    # Train
    train_res = train_one_epoch(model, criterion, optimizer, train_loader, DEVICE)

    # Validate (dengan pembaruan bobot adaptif)
    val_res = evaluate(model, criterion, val_loader, DEVICE, update_weights=True)

    # Scheduler step
    scheduler.step(val_res['loss'])
    current_lr = optimizer.param_groups[0]['lr']

    elapsed = time.time() - t0
    avg_val_mae = (val_res['mae_sbp'] + val_res['mae_dbp'] + val_res['mae_map']) / 3.0

    # Simpan history
    history['train_loss'].append(train_res['loss'])
    history['val_loss'].append(val_res['loss'])
    history['val_mae_sbp'].append(val_res['mae_sbp'])
    history['val_mae_dbp'].append(val_res['mae_dbp'])
    history['val_mae_map'].append(val_res['mae_map'])
    history['lr'].append(current_lr)
    history['weights'].append(criterion.weights.tolist())

    # Best model checkpoint
    is_best = avg_val_mae < best_val_mae
    if is_best:
        best_val_mae = avg_val_mae
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_mae': best_val_mae,
            'history': history,
        }, best_model_path)
        mark = '★ BEST'
    else:
        mark = ''

    w = criterion.weights.tolist()
    print(
        f'Epoch [{epoch:02d}/{NUM_EPOCHS}] ({elapsed:.1f}s) | '
        f'Train Loss: {train_res["loss"]:.4f} | '
        f'Val Loss: {val_res["loss"]:.4f} | '
        f'MAE → SBP: {val_res["mae_sbp"]:.2f}, DBP: {val_res["mae_dbp"]:.2f}, '
        f'MAP: {val_res["mae_map"]:.2f} mmHg {mark} | '
        f'θ=[{w[0]:.2f},{w[1]:.2f},{w[2]:.2f}] | LR={current_lr:.2e}'
    )

total_time = time.time() - start_time
print('\n' + '=' * 80)
print(f'Pelatihan selesai dalam {total_time / 60:.1f} menit ({total_time / 3600:.2f} jam).')
print(f'Model terbaik (Avg Val MAE: {best_val_mae:.2f} mmHg) disimpan di: {best_model_path}')
print('=' * 80)

---
## 11. Visualisasi Kurva Training

In [ ]:
epochs_arr = np.arange(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# ── Panel 1: Total Loss ──
axes[0, 0].plot(epochs_arr, history['train_loss'], 'o-', label='Train Loss', color='#1f77b4', lw=2)
axes[0, 0].plot(epochs_arr, history['val_loss'], 's--', label='Val Loss', color='#ff7f0e', lw=2)
axes[0, 0].set_title('Kurva Loss (Train vs Validation)', fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss (Weighted MAE)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# ── Panel 2: MAE per Target ──
axes[0, 1].plot(epochs_arr, history['val_mae_sbp'], 'o-', label='SBP MAE', color='#e74c3c', lw=2)
axes[0, 1].plot(epochs_arr, history['val_mae_dbp'], 's-', label='DBP MAE', color='#3498db', lw=2)
axes[0, 1].plot(epochs_arr, history['val_mae_map'], '^-', label='MAP MAE', color='#2ecc71', lw=2)
axes[0, 1].axhline(y=5.0, color='gray', linestyle=':', alpha=0.7, label='Target AAMI ≤ 5 mmHg')
axes[0, 1].set_title('Validation MAE per Target BP', fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('MAE (mmHg)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# ── Panel 3: Learning Rate ──
axes[1, 0].plot(epochs_arr, history['lr'], 'D-', color='#9b59b6', lw=2)
axes[1, 0].set_title('Learning Rate Schedule', fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Learning Rate')
axes[1, 0].set_yscale('log')
axes[1, 0].grid(True, alpha=0.3)

# ── Panel 4: Adaptive Task Weights ──
weights_arr = np.array(history['weights'])
axes[1, 1].plot(epochs_arr, weights_arr[:, 0], 'o-', label='θ_SBP', color='#e74c3c', lw=2)
axes[1, 1].plot(epochs_arr, weights_arr[:, 1], 's-', label='θ_DBP', color='#3498db', lw=2)
axes[1, 1].plot(epochs_arr, weights_arr[:, 2], '^-', label='θ_MAP', color='#2ecc71', lw=2)
axes[1, 1].axhline(y=1/3, color='gray', linestyle=':', alpha=0.5, label='Bobot Awal (1/3)')
axes[1, 1].set_title('Evolusi Bobot Adaptif (θ) per Epoch', fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Bobot θ')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Monitoring Pelatihan Multi-Task BiLSTM', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Grafik disimpan: /kaggle/working/training_curves.png')

---
## 12. Evaluasi Akhir pada Test Set

In [ ]:
# Muat model terbaik
checkpoint = torch.load(best_model_path, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f'Model terbaik dimuat dari epoch {checkpoint["epoch"]} '
      f'(Best Avg Val MAE: {checkpoint["best_val_mae"]:.2f} mmHg)')

# Kumpulkan semua prediksi dan ground truth dari test set
all_preds = []
all_trues = []

with torch.no_grad():
    for x_b, y_b in tqdm(test_loader, desc='Evaluasi Test Set'):
        x_b = x_b.to(DEVICE)
        preds = model.predict_all(x_b)  # [batch, 3]
        all_preds.append(preds.cpu().numpy())
        all_trues.append(y_b.numpy())

all_preds = np.concatenate(all_preds, axis=0)  # [N_test, 3]
all_trues = np.concatenate(all_trues, axis=0)  # [N_test, 3]

# Metrik evaluasi
errors = all_preds - all_trues  # Error = Prediksi - Ground Truth
abs_errors = np.abs(errors)

target_names = ['SBP', 'DBP', 'MAP']
print('\n' + '=' * 70)
print('  HASIL EVALUASI AKHIR PADA TEST SET (DATA UNSEEN)')
print('  Standar Evaluasi: AAMI SP10 & BHS Protocol')
print('=' * 70)

results = {}
for i, name in enumerate(target_names):
    mae = np.mean(abs_errors[:, i])
    std = np.std(errors[:, i])
    me = np.mean(errors[:, i])  # Mean Error (bias)
    rmse = np.sqrt(np.mean(errors[:, i] ** 2))
    results[name] = {'MAE': mae, 'STD': std, 'ME': me, 'RMSE': rmse}

    # BHS Grading
    pct_5 = np.mean(abs_errors[:, i] <= 5) * 100
    pct_10 = np.mean(abs_errors[:, i] <= 10) * 100
    pct_15 = np.mean(abs_errors[:, i] <= 15) * 100

    if pct_5 >= 60 and pct_10 >= 85 and pct_15 >= 95:
        bhs_grade = 'A'
    elif pct_5 >= 50 and pct_10 >= 75 and pct_15 >= 90:
        bhs_grade = 'B'
    elif pct_5 >= 40 and pct_10 >= 65 and pct_15 >= 85:
        bhs_grade = 'C'
    else:
        bhs_grade = 'D'

    aami_pass = '✅ LULUS' if mae <= 5.0 and std <= 8.0 else '❌ GAGAL'

    print(f'\n  {name}:')
    print(f'    MAE       : {mae:.2f} mmHg')
    print(f'    Mean Error: {me:+.2f} ± {std:.2f} mmHg')
    print(f'    RMSE      : {rmse:.2f} mmHg')
    print(f'    BHS       : ≤5mmHg={pct_5:.1f}% | ≤10mmHg={pct_10:.1f}% | ≤15mmHg={pct_15:.1f}% → Grade {bhs_grade}')
    print(f'    AAMI SP10 : MAE≤5 & STD≤8 → {aami_pass}')

avg_mae = np.mean([r['MAE'] for r in results.values()])
print(f'\n  Rata-rata MAE Keseluruhan: {avg_mae:.2f} mmHg')
print('=' * 70)

---
## 13. Scatter Plot: Prediksi vs Ground Truth

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#e74c3c', '#3498db', '#2ecc71']

for i, (name, color) in enumerate(zip(target_names, colors)):
    true_vals = all_trues[:, i]
    pred_vals = all_preds[:, i]
    mae = results[name]['MAE']

    axes[i].scatter(true_vals, pred_vals, alpha=0.15, s=8, color=color, edgecolors='none')

    # Garis ideal y = x
    lims = [min(true_vals.min(), pred_vals.min()) - 5,
            max(true_vals.max(), pred_vals.max()) + 5]
    axes[i].plot(lims, lims, 'k--', lw=1.5, alpha=0.7, label='Ideal (y=x)')

    # Band ±5 mmHg
    axes[i].fill_between(lims, [l - 5 for l in lims], [l + 5 for l in lims],
                         alpha=0.1, color='green', label='±5 mmHg')

    axes[i].set_title(f'{name} — Prediksi vs Ground Truth\nMAE: {mae:.2f} mmHg',
                      fontweight='bold', fontsize=12)
    axes[i].set_xlabel(f'{name} Ground Truth (mmHg)')
    axes[i].set_ylabel(f'{name} Prediksi (mmHg)')
    axes[i].set_xlim(lims)
    axes[i].set_ylim(lims)
    axes[i].set_aspect('equal')
    axes[i].legend(loc='upper left', fontsize=9)
    axes[i].grid(True, alpha=0.3)

plt.suptitle(f'Prediksi vs Ground Truth — Test Set ({len(all_trues):,} sampel)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/kaggle/working/scatter_pred_vs_true.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Scatter plot disimpan: /kaggle/working/scatter_pred_vs_true.png')

---
## 14. Bland-Altman Plot (Analisis Agreement Klinis)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (name, color) in enumerate(zip(target_names, colors)):
    true_vals = all_trues[:, i]
    pred_vals = all_preds[:, i]

    mean_vals = (true_vals + pred_vals) / 2
    diff_vals = pred_vals - true_vals

    mean_diff = np.mean(diff_vals)
    std_diff = np.std(diff_vals)
    loa_upper = mean_diff + 1.96 * std_diff
    loa_lower = mean_diff - 1.96 * std_diff

    axes[i].scatter(mean_vals, diff_vals, alpha=0.1, s=6, color=color, edgecolors='none')
    axes[i].axhline(mean_diff, color='black', linestyle='-', lw=1.5,
                    label=f'Bias: {mean_diff:+.2f} mmHg')
    axes[i].axhline(loa_upper, color='red', linestyle='--', lw=1,
                    label=f'+1.96 SD: {loa_upper:+.2f}')
    axes[i].axhline(loa_lower, color='red', linestyle='--', lw=1,
                    label=f'−1.96 SD: {loa_lower:+.2f}')
    axes[i].axhline(0, color='gray', linestyle=':', alpha=0.5)

    axes[i].set_title(f'Bland-Altman: {name}', fontweight='bold', fontsize=12)
    axes[i].set_xlabel(f'Mean ({name} Ground Truth + Prediksi) / 2 (mmHg)')
    axes[i].set_ylabel(f'Perbedaan (Prediksi − Ground Truth) (mmHg)')
    axes[i].legend(fontsize=8, loc='upper right')
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Bland-Altman Agreement Analysis — Test Set',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/kaggle/working/bland_altman.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Bland-Altman disimpan: /kaggle/working/bland_altman.png')

---
## 15. Simpan Hasil & Model Final

In [ ]:
# Simpan riwayat training
history_path = '/kaggle/working/training_history.json'
with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)
print(f'📁 Training history: {history_path}')

# Simpan hasil evaluasi
eval_results = {
    'test_samples': len(all_trues),
    'results': {name: {k: float(v) for k, v in r.items()} for name, r in results.items()},
    'avg_mae': float(avg_mae),
    'best_epoch': checkpoint['epoch'],
    'hyperparameters': {
        'batch_size': BATCH_SIZE,
        'num_epochs': NUM_EPOCHS,
        'learning_rate': LEARNING_RATE,
        'hidden_size': HIDDEN_SIZE,
        'num_layers': NUM_LAYERS,
        'lstm_dropout': DROPOUT,
        'fc_dropout': FC_DROPOUT,
        'grad_clip': GRAD_CLIP,
        'optimizer': 'AdamW',
        'scheduler': f'ReduceLROnPlateau(factor={SCHEDULER_FACTOR}, patience={SCHEDULER_PATIENCE})',
    }
}
eval_path = '/kaggle/working/evaluation_results.json'
with open(eval_path, 'w') as f:
    json.dump(eval_results, f, indent=2)
print(f'📁 Evaluation results: {eval_path}')

# Simpan model final (lengkap untuk reproduksi)
final_path = '/kaggle/working/final_model_full.pt'
torch.save({
    'model_state_dict': model.state_dict(),
    'model_config': {
        'input_size': INPUT_SIZE,
        'hidden_size': HIDDEN_SIZE,
        'num_layers': NUM_LAYERS,
        'lstm_dropout': DROPOUT,
        'fc_dropout': FC_DROPOUT,
    },
    'preprocessing_config': {
        'fs': FS,
        'seq_len': SEQ_LEN,
        'notch_freq': NOTCH_FREQ,
        'notch_q': NOTCH_Q,
        'bp_lowcut': BP_LOWCUT,
        'bp_highcut': BP_HIGHCUT,
        'bp_order': BP_ORDER,
        'normalization': 'minmax [0,1]',
    },
    'evaluation': eval_results,
}, final_path)
print(f'📁 Final model: {final_path}')

print('\n✅ Semua artefak telah disimpan di /kaggle/working/')
print('   Unduh dari panel Output di sebelah kanan Kaggle notebook.')

---
## 16. Ringkasan File Output

| File | Deskripsi |
|---|---|
| `checkpoints/best_model.pt` | Checkpoint model terbaik (berdasarkan Val MAE terendah) |
| `final_model_full.pt` | Model + config + hasil evaluasi (untuk deployment) |
| `training_history.json` | Riwayat loss, MAE, LR, dan bobot adaptif per epoch |
| `evaluation_results.json` | Metrik evaluasi akhir (MAE, RMSE, BHS, AAMI) |
| `training_curves.png` | Grafik kurva konvergensi training |
| `scatter_pred_vs_true.png` | Scatter plot prediksi vs ground truth |
| `bland_altman.png` | Bland-Altman agreement analysis |